# Proof of concept: grounded interview

One NVIDIA persona answers the same personal-finance questions three ways:

1. **Baseline.** Identity only. No emotional journey.
2. **Short journey.** One sentence about how their mood moves. They are told not to announce it.
3. **Detailed journey.** That sentence, plus the real sentiment beats and two or three transcript lines used only as manner.

The four journey shapes are a **provisional** fold of the six clusters in `clustering_emotional_journeys.ipynb`.
Warm-up stays its own group. The two positive dip-and-recover groups fold together. The middle peak stays. The two falling groups fold together.
The example video for each shape is the one whose sentiment curve is closest to that group's average curve.

Replace the stand-ins when they show up:

- Wenjun's journey picks: a CSV with columns `shape` and `video_id` at `clustering_outputs/selected_journeys.csv`.
- Prakhar's questions: a CSV with a `question` column at `Dataset/interview_questions.csv`.

The notebook reads only small files. It does not download the `.csd` feature files or the full NVIDIA dump.

Answers are generated with Azure OpenAI. Fill in `endpoint`, `subscription_key`, and `api_version` in the generation cell.
With those left blank, the notebook still writes the prompts and leaves the answer cells blank.

## 1. Setup

In [1]:
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print('Colab:', IN_COLAB)

Mounted at /content/drive
Colab: True


In [2]:
import json
import os
import urllib.error
import urllib.request

import numpy as np
import pandas as pd

RANDOM_STATE = 42
L = 12
GRID = np.linspace(0, 1, L)

# First API run can be one shape. Use all four names to generate every variant.
SHAPES_TO_RUN = ['warms_up', 'positive_dip_recover', 'mid_peak', 'cools_off']

DRIVE_ROOTS = []
if IN_COLAB:
    DRIVE_ROOTS = [
        '/content/drive/MyDrive/Northwestern x Accenture',
        '/content/drive/My Drive/Northwestern x Accenture',
    ]
LOCAL_ROOTS = ['.', './data', '..', '../data']

def first_existing(relative_options):
    for root in DRIVE_ROOTS + LOCAL_ROOTS:
        for rel in relative_options:
            path = os.path.join(root, rel)
            if os.path.exists(path):
                return path
    return None

PERSONA_CSV = first_existing([
    'Dataset/Nvidia_Nemotron-Personas-USA-sample-30.csv',
    'Nvidia_Nemotron-Personas-USA-sample-30.csv',
])
ASSIGN_CSV = first_existing([
    'Dataset/CMU-MOSEI/clustering_outputs/cluster_assignments.csv',
    'CMU-MOSEI/clustering_outputs/cluster_assignments.csv',
    'clustering_outputs/cluster_assignments.csv',
])
SUMMARY_CSV = first_existing([
    'Dataset/CMU-MOSEI/clustering_outputs/cluster_summaries.csv',
    'CMU-MOSEI/clustering_outputs/cluster_summaries.csv',
    'clustering_outputs/cluster_summaries.csv',
])
TRANSCRIPT_CSV = first_existing([
    'Dataset/CMU-MOSEI/segment_transcripts_with_labels.csv',
    'CMU-MOSEI/segment_transcripts_with_labels.csv',
    'segment_transcripts_with_labels.csv',
])
JOURNEY_OVERRIDE_CSV = first_existing([
    'Dataset/CMU-MOSEI/clustering_outputs/selected_journeys.csv',
    'CMU-MOSEI/clustering_outputs/selected_journeys.csv',
    'clustering_outputs/selected_journeys.csv',
])
QUESTION_OVERRIDE_CSV = first_existing([
    'Dataset/interview_questions.csv',
    'interview_questions.csv',
    'PMO/interview_questions.csv',
])

missing = [name for name, path in [
    ('persona sample', PERSONA_CSV),
    ('cluster assignments', ASSIGN_CSV),
    ('transcripts', TRANSCRIPT_CSV),
] if path is None]
assert not missing, 'Missing files: ' + ', '.join(missing) + '. Mount Drive or set the path variables.'

if IN_COLAB:
    OUTPUT_DIR = os.path.join(os.path.dirname(TRANSCRIPT_CSV), 'poc_outputs')
    if '/CMU-MOSEI/' not in OUTPUT_DIR.replace('\\', '/'):
        dataset_root = os.path.dirname(os.path.dirname(TRANSCRIPT_CSV))
        OUTPUT_DIR = os.path.join(dataset_root, 'CMU-MOSEI', 'poc_outputs')
else:
    OUTPUT_DIR = './poc_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Persona:', PERSONA_CSV)
print('Assignments:', ASSIGN_CSV)
print('Transcripts:', TRANSCRIPT_CSV)
print('Journey override:', JOURNEY_OVERRIDE_CSV or '(none, using the provisional fold)')
print('Question override:', QUESTION_OVERRIDE_CSV or '(none, using placeholders)')
print('Outputs:', OUTPUT_DIR)

Persona: /content/drive/MyDrive/Northwestern x Accenture/Dataset/Nvidia_Nemotron-Personas-USA-sample-30.csv
Assignments: /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/clustering_outputs/cluster_assignments.csv
Transcripts: /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/segment_transcripts_with_labels.csv
Journey override: (none, using the provisional fold)
Question override: (none, using placeholders)
Outputs: /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/poc_outputs


## 2. Four provisional journey shapes

Each cleaned video is resampled onto 12 points, the same way as the clustering notebook.
A folded group's average curve is the mean of its members. The example video is the member closest to that mean in ordinary distance, on the real sentiment curve.

In [3]:
GROUPS = [
    {
        'shape': 'warms_up',
        'source_clusters': [0],
        'short': (
            'Start near neutral and gradually warm up over the course of the interview. '
            'Let that show in your tone. Do not announce this pattern.'
        ),
    },
    {
        'shape': 'positive_dip_recover',
        'source_clusters': [1, 4],
        'short': (
            'Start positive, grow more cautious in the middle, then recover to a positive close. '
            'Do not announce this pattern.'
        ),
    },
    {
        'shape': 'mid_peak',
        'source_clusters': [3],
        'short': (
            'Start near neutral, become more engaged and positive in the middle, then ease off slightly by the end. '
            'Do not announce this pattern.'
        ),
    },
    {
        'shape': 'cools_off',
        'source_clusters': [2, 5],
        'short': (
            'Start mildly positive or neutral, then grow more skeptical and negative as the interview goes on. '
            'Do not announce this pattern.'
        ),
    },
]

def resample(times, values, grid):
    order = np.argsort(times)
    t = np.asarray(times, float)[order]
    v = np.asarray(values, float)[order]
    uniq, idx = np.unique(t, return_inverse=True)
    if len(uniq) < len(t):
        v = np.array([v[idx == k].mean() for k in range(len(uniq))])
        t = uniq
    if len(t) == 1 or t.max() == t.min():
        return np.full(len(grid), float(np.mean(v)))
    t_norm = (t - t.min()) / (t.max() - t.min())
    return np.interp(grid, t_norm, v)

assignments = pd.read_csv(ASSIGN_CSV)
transcripts = pd.read_csv(TRANSCRIPT_CSV)
transcripts = transcripts[transcripts['video_id'].isin(set(assignments['video_id']))].copy()
transcripts['mid'] = (transcripts['interval_start'] + transcripts['interval_end']) / 2.0

video_rows = {}
arcs = {}
for video_id, group in transcripts.groupby('video_id'):
    group = group.sort_values('mid')
    video_rows[video_id] = group
    arcs[video_id] = resample(group['mid'].to_numpy(), group['sentiment'].to_numpy(), GRID)

journey_rows = []
for group in GROUPS:
    members = assignments[assignments['cluster'].isin(group['source_clusters'])]['video_id'].tolist()
    member_arcs = np.vstack([arcs[video_id] for video_id in members])
    mean_arc = member_arcs.mean(axis=0)
    distances = np.linalg.norm(member_arcs - mean_arc, axis=1)
    chosen_id = members[int(np.argmin(distances))]
    chosen_rows = video_rows[chosen_id]
    scores = [round(float(x), 2) for x in chosen_rows['sentiment'].tolist()]
    snippets = chosen_rows['transcript'].fillna('').astype(str).tolist()
    picks = [snippets[0], snippets[len(snippets) // 2], snippets[-1]]
    picks = [text.strip()[:160] for text in picks if text.strip()]
    # keep at most three, and drop exact duplicates while preserving order
    seen = []
    for text in picks:
        if text not in seen:
            seen.append(text)
    journey_rows.append({
        'shape': group['shape'],
        'source_clusters': ','.join(str(c) for c in group['source_clusters']),
        'video_id': chosen_id,
        'n_utterances': int(len(chosen_rows)),
        'distance_to_mean': float(distances.min()),
        'sentiment_sequence': scores,
        'snippets': seen[:3],
        'short': group['short'],
        'provisional': True,
    })

if JOURNEY_OVERRIDE_CSV:
    override = pd.read_csv(JOURNEY_OVERRIDE_CSV)
    required = {'shape', 'video_id'}
    assert required <= set(override.columns), 'selected_journeys.csv needs shape and video_id columns'
    by_shape = {row['shape']: row for row in journey_rows}
    for record in override.to_dict(orient='records'):
        shape = record['shape']
        video_id = str(record['video_id'])
        assert shape in by_shape, f'Unknown shape in override: {shape}'
        assert video_id in video_rows, f'Override video not in the transcript table: {video_id}'
        chosen_rows = video_rows[video_id]
        scores = [round(float(x), 2) for x in chosen_rows['sentiment'].tolist()]
        snippets = [text.strip()[:160] for text in chosen_rows['transcript'].fillna('').astype(str).tolist() if str(text).strip()]
        picks = []
        for text in [snippets[0], snippets[len(snippets) // 2], snippets[-1]]:
            if text not in picks:
                picks.append(text)
        by_shape[shape].update({
            'video_id': video_id,
            'n_utterances': int(len(chosen_rows)),
            'distance_to_mean': None,
            'sentiment_sequence': scores,
            'snippets': picks[:3],
            'provisional': False,
        })
    journey_rows = [by_shape[group['shape']] for group in GROUPS]
    print('Using Wenjun override:', JOURNEY_OVERRIDE_CSV)
else:
    print('Provisional fold of clusters 0 | 1+4 | 3 | 2+5. Swap in selected_journeys.csv when it exists.')

journeys = pd.DataFrame(journey_rows)
print(journeys[['shape', 'source_clusters', 'video_id', 'n_utterances', 'distance_to_mean', 'provisional']].to_string(index=False))
for row in journey_rows:
    print(f"\n{row['shape']}  {row['video_id']}  sentiment={row['sentiment_sequence']}")
    print(' ', row['short'])

Provisional fold of clusters 0 | 1+4 | 3 | 2+5. Swap in selected_journeys.csv when it exists.
               shape source_clusters    video_id  n_utterances  distance_to_mean  provisional
            warms_up               0 etPDgRjrnJ8             4          0.580726         True
positive_dip_recover             1,4 TXiOSZdaLJ8             3          0.447576         True
            mid_peak               3 1olbQVO_C-8             3          0.574000         True
           cools_off             2,5 v9gjg7TDlTA             5          0.479959         True

warms_up  etPDgRjrnJ8  sentiment=[0.0, 0.0, 1.0, 1.0]
  Start near neutral and gradually warm up over the course of the interview. Let that show in your tone. Do not announce this pattern.

positive_dip_recover  TXiOSZdaLJ8  sentiment=[0.67, 0.0, 0.33]
  Start positive, grow more cautious in the middle, then recover to a positive close. Do not announce this pattern.

mid_peak  1olbQVO_C-8  sentiment=[0.0, 0.67, 0.0]
  Start near ne

## 3. One persona and the placeholder questions

The persona is one working-age adult from the 30-person NVIDIA sample.
A manager is preferred when the sample has one, because the scenario is a household decision about a bank app.
The text kept for the prompt is the short `persona` field plus one role line. The other persona columns are left out.

The five questions are placeholders for a regional bank's personal-finance app.
They are replaced automatically if `Dataset/interview_questions.csv` is present.

In [4]:
personas = pd.read_csv(PERSONA_CSV)
working = personas[
    personas['age'].between(21, 65)
    & ~personas['occupation'].isin(['not_in_workforce', 'no_occupation'])
].copy()
assert len(working), 'No working-age employed persona in the sample.'
managers = working[working['occupation'].str.contains('manager', case=False, na=False)]
persona_row = (managers if len(managers) else working).iloc[0]

brief = str(persona_row['persona']).strip()
if len(brief) > 450:
    brief = brief[:450].rsplit(' ', 1)[0] + '.'
role = str(persona_row['occupation']).replace('_', ' ')
PERSONA_TEXT = (
    f"{brief} Role: {role}, age {int(persona_row['age'])}, "
    f"{persona_row['city']}, {persona_row['state']}."
)
print('Persona uuid:', persona_row['uuid'])
print(PERSONA_TEXT)

PLACEHOLDER_QUESTIONS = [
    'What is your first reaction to a regional bank app that automatically sorts your spending and suggests a weekly budget?',
    'How would this fit the way you handle money today?',
    'What would make you not trust it with your account data?',
    'Would you pay a few dollars a month for it, and why?',
    'What would you need to see before you recommended it to someone like you?',
]

if QUESTION_OVERRIDE_CSV:
    loaded = pd.read_csv(QUESTION_OVERRIDE_CSV)
    assert 'question' in loaded.columns, 'interview_questions.csv needs a question column'
    QUESTIONS = [str(q).strip() for q in loaded['question'].tolist() if str(q).strip()]
    QUESTION_SOURCE = 'override:' + QUESTION_OVERRIDE_CSV
else:
    QUESTIONS = PLACEHOLDER_QUESTIONS
    QUESTION_SOURCE = 'placeholder: regional-bank personal-finance app, until Prakhar uploads interview_questions.csv'
print('Questions:', QUESTION_SOURCE)
for i, question in enumerate(QUESTIONS, 1):
    print(f'{i}. {question}')

Persona uuid: 0ee14f548c2c48c99b6ed3d8369e194b
Filomena Nguyen navigates life with data‑driven curiosity and a flexible planning style that sometimes spirals into last‑minute scrambles, counterbalanced by regular yoga, a love for spontaneous hikes, and a habit of worrying about both work and personal projects. Role: general or operations manager, age 30, Parsons, WV.
Questions: placeholder: regional-bank personal-finance app, until Prakhar uploads interview_questions.csv
1. What is your first reaction to a regional bank app that automatically sorts your spending and suggests a weekly budget?
2. How would this fit the way you handle money today?
3. What would make you not trust it with your account data?
4. Would you pay a few dollars a month for it, and why?
5. What would you need to see before you recommended it to someone like you?


## 4. Build the three prompts

`SHAPES_TO_RUN` at the top chooses which shapes are sent to the model. Prompt text for the selected shapes is always saved, even when no API key is set.

In [5]:
def journey_block(row, detail):
    if detail == 'baseline':
        return ''
    if detail == 'short':
        body = row['short']
    else:
        beats = ', '.join(f'{score:+.2f}' for score in row['sentiment_sequence'])
        lines = '\n'.join(f'- {text}' for text in row['snippets'])
        body = (
            f"{row['short']}\n\n"
            'Your mood across the conversation, as sentiment scores from -3 (very negative) to +3 (very positive), '
            f'in order: {beats}.\n\n'
            'These lines are examples of manner from a different conversation. '
            'Copy the manner, not the subject. Do not talk about the subject of these lines:\n'
            f'{lines}'
        )
    return (
        'How you tend to react during a conversation:\n'
        f'{body}\n'
        'Let this shape the tone and pacing of your answers. Do not state the pattern outright, and do not change who you are.'
    )

def build_prompt(persona_text, question, journey_row, detail):
    block = journey_block(journey_row, detail)
    journey_part = f'\n{block}\n' if block else '\n'
    return (
        'You are taking part in a product interview. Stay in character. Answer as this person would, in first person. '
        'Do not mention that you are an AI, a persona, or that you were given a mood instruction.\n\n'
        f'Who you are:\n{persona_text}'
        f'{journey_part}\n'
        f'Question:\n{question}\n\n'
        'Answer in 3 to 6 sentences.'
    )

selected = [row for row in journey_rows if row['shape'] in SHAPES_TO_RUN]
unknown = set(SHAPES_TO_RUN) - {row['shape'] for row in journey_rows}
assert not unknown, f'Unknown shapes in SHAPES_TO_RUN: {unknown}'
assert selected, 'SHAPES_TO_RUN is empty.'

prompt_rows = []
for row in selected:
    for detail in ('baseline', 'short', 'detailed'):
        for index, question in enumerate(QUESTIONS, 1):
            prompt_rows.append({
                'shape': row['shape'],
                'video_id': row['video_id'],
                'detail': detail,
                'question_index': index,
                'question': question,
                'question_source': QUESTION_SOURCE,
                'prompt': build_prompt(PERSONA_TEXT, question, row, detail),
            })
prompts = pd.DataFrame(prompt_rows)
print(f'{len(prompts)} prompts for shapes: {[row["shape"] for row in selected]}')
print('\n--- sample short prompt ---\n')
print(prompts.loc[prompts['detail'].eq('short'), 'prompt'].iloc[0])

15 prompts for shapes: ['warms_up']

--- sample short prompt ---

You are taking part in a product interview. Stay in character. Answer as this person would, in first person. Do not mention that you are an AI, a persona, or that you were given a mood instruction.

Who you are:
Filomena Nguyen navigates life with data‑driven curiosity and a flexible planning style that sometimes spirals into last‑minute scrambles, counterbalanced by regular yoga, a love for spontaneous hikes, and a habit of worrying about both work and personal projects. Role: general or operations manager, age 30, Parsons, WV.
How you tend to react during a conversation:
Start near neutral and gradually warm up over the course of the interview. Let that show in your tone. Do not announce this pattern.
Let this shape the tone and pacing of your answers. Do not state the pattern outright, and do not change who you are.

Question:
What is your first reaction to a regional bank app that automatically sorts your spending an

## 5. Generate answers with Azure OpenAI

Paste your Azure values into the blank strings below. `deployment` is already set to `gpt-4o`.
Leave the key out of git. With the key or endpoint still blank, answers stay empty and the prompts are still saved.

In [6]:
endpoint = ""
model_name = "gpt-4o"
deployment = "gpt-4o"
subscription_key = ""
api_version = "2024-12-01-preview"

def post_json(url, payload, headers):
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read().decode("utf-8"))

def azure_ready():
    missing = [
        name for name, value in [
            ("endpoint", endpoint),
            ("deployment", deployment),
            ("subscription_key", subscription_key),
            ("api_version", api_version),
        ] if not str(value).strip()
    ]
    return missing

def generate(prompt):
    base = endpoint.rstrip("/")
    url = f"{base}/openai/deployments/{deployment}/chat/completions?api-version={api_version}"
    payload = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7,
    }
    if model_name.strip():
        payload["model"] = model_name.strip()
    body = post_json(
        url,
        payload,
        {"api-key": subscription_key, "Content-Type": "application/json"},
    )
    return body["choices"][0]["message"]["content"].strip()

answers = []
errors = []
missing_azure = azure_ready()
if missing_azure:
    print("Azure settings still blank:", ", ".join(missing_azure) + ". Prompts will be saved and answers left blank.")
else:
    label = f"Generating with Azure deployment {deployment}"
    if model_name.strip():
        label += f" ({model_name})"
    print(label + ".")

for record in prompts.to_dict(orient="records"):
    answer = ""
    error = ""
    if not missing_azure:
        try:
            answer = generate(record["prompt"])
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            errors.append(error)
    answers.append(answer)
    prompts.loc[len(answers) - 1, "answer"] = answer
    prompts.loc[len(answers) - 1, "error"] = error

if "answer" not in prompts.columns:
    prompts["answer"] = answers
    prompts["error"] = ""
print("answers filled:", int(prompts["answer"].astype(str).str.len().gt(0).sum()), "of", len(prompts))
if errors:
    print("first error:", errors[0])


Generating with Azure deployment gpt-4o (gpt-4o).
answers filled: 15 of 15


## 6. Save the comparison table

In [7]:
pieces = []
for shape, group in prompts.groupby('shape', sort=False):
    wide = {}
    for detail, detail_group in group.groupby('detail', sort=False):
        detail_group = detail_group.sort_values('question_index')
        wide[detail] = detail_group
    base = wide['baseline'][['question_index', 'question', 'question_source']].reset_index(drop=True)
    base['shape'] = shape
    base['video_id'] = group['video_id'].iloc[0]
    base['baseline_answer'] = wide['baseline'].sort_values('question_index')['answer'].tolist()
    base['short_answer'] = wide['short'].sort_values('question_index')['answer'].tolist()
    base['detailed_answer'] = wide['detailed'].sort_values('question_index')['answer'].tolist()
    pieces.append(base)
results = pd.concat(pieces, ignore_index=True)

export_journeys = journeys.copy()
export_journeys['sentiment_sequence'] = export_journeys['sentiment_sequence'].apply(
    lambda scores: ';'.join(str(score) for score in scores)
)
export_journeys['snippets'] = export_journeys['snippets'].apply(lambda lines: ' || '.join(lines))

journey_path = os.path.join(OUTPUT_DIR, 'provisional_journeys.csv')
prompt_path = os.path.join(OUTPUT_DIR, 'prompts.csv')
result_path = os.path.join(OUTPUT_DIR, 'interview_results.csv')
export_journeys.to_csv(journey_path, index=False)
prompts.drop(columns=['prompt']).to_csv(os.path.join(OUTPUT_DIR, 'prompt_index.csv'), index=False)
prompts.to_csv(prompt_path, index=False)
results.to_csv(result_path, index=False)
print('wrote', journey_path)
print('wrote', prompt_path)
print('wrote', result_path)
results[['shape', 'question_index', 'question']]

wrote /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/poc_outputs/provisional_journeys.csv
wrote /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/poc_outputs/prompts.csv
wrote /content/drive/MyDrive/Northwestern x Accenture/Dataset/CMU-MOSEI/poc_outputs/interview_results.csv


,shape,question_index,question
0,warms_up,1,What is your first reaction to a regional bank...
1,warms_up,2,How would this fit the way you handle money to...
2,warms_up,3,What would make you not trust it with your acc...
3,warms_up,4,"Would you pay a few dollars a month for it, an..."
4,warms_up,5,What would you need to see before you recommen...


## Notes

- The four shapes are provisional until `selected_journeys.csv` appears. The fold is clusters 0, 1 with 4, 3, and 2 with 5.
- `SHAPES_TO_RUN` lists all four shapes, so a full run interviews each one at both detail levels plus the no-journey baseline. The same prompt function is used every time.
- Questions stay marked as placeholders until `interview_questions.csv` is on Drive.
- This notebook does not re-cluster, does not sample the full NVIDIA file, and does not write the class project plan.